<a href="https://colab.research.google.com/github/kuteesatendojeremiah/DS_intern_work/blob/main/project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS Intern Work — Project 2: Fraud Detection Pipeline (Supervised Learning)

Goal: train and tune a classifier to catch fraudulent transactions in a highly imbalanced dataset (~99.83% legitimate / ~0.17% fraud), using the [Kaggle Credit Card Fraud Detection dataset](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud).

**Zero-Leakage Protocol:**
- Stratified train/test split first, before anything else touches the data.
- SMOTE and scaling live *inside* the pipeline, so cross-validation only ever resamples/scales the training fold, never the validation fold, never the test set.
- Test set stays untouched (real-world imbalance) until final evaluation.
- No accuracy metric. Precision, Recall, ROC-AUC, and a Confusion Matrix on the held-out test set only.

In [ ]:
# Install dependencies (imbalanced-learn for SMOTE + imblearn.pipeline, kagglehub for dataset download)
!pip install imbalanced-learn kagglehub -q

## 1. Load the data

Downloads the dataset straight from Kaggle via `kagglehub`. On first run in Colab this will prompt you to authenticate (either `kagglehub.login()` interactively, or upload a `kaggle.json` API token beforehand, see the Kaggle API docs).

In [ ]:
import os
import kagglehub

dataset_path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
print("Dataset downloaded to:", dataset_path)
print(os.listdir(dataset_path))

In [ ]:
import pandas as pd

csv_path = os.path.join(dataset_path, "creditcard.csv")
df = pd.read_csv(csv_path)

print("Shape:", df.shape)
df.head()

## EDA: missing values, dtypes, duplicates

Quick data-integrity pass before anything else touches the data. `V1`-`V28` are already PCA components (anonymized), so there's no raw categorical cleanup to do here, but we still check for nulls, dtype issues, and duplicate rows rather than assume they're absent.

In [ ]:
df.info()
print()
print("Missing values per column:")
print(df.isnull().sum().sum(), "total missing values")
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
df.describe()

In [ ]:
# Duplicate rows: this dataset is known to contain exact duplicates
n_dupes = df.duplicated().sum()
print(f"Duplicate rows: {n_dupes} ({n_dupes / len(df):.4%} of data)")

Duplicates matter here specifically because of the zero-leakage protocol: if a duplicate row exists, the split could place one copy in `X_train` and its exact twin in `X_test`, which is leakage (the model would effectively be evaluated on a row it already trained on). Dropping duplicates now, before the split, is the correct place to do it.

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after dropping duplicates:", df.shape)

In [ ]:
# Confirm the class imbalance (after deduping)
class_counts = df["Class"].value_counts()
class_pct = df["Class"].value_counts(normalize=True) * 100

print(class_counts)
print()
print(class_pct.round(4))

## 2. Stratified train/test split, before any processing

This is the only split in the entire notebook. Everything downstream (scaling, SMOTE, hyperparameter search) happens only on `X_train`/`y_train`. `X_test`/`y_test` stay untouched and keep the real-world 99.83/0.17 imbalance until final evaluation.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train shape:", X_train.shape, "Fraud rate:", y_train.mean())
print("Test shape: ", X_test.shape, "Fraud rate:", y_test.mean())

## 3. Build the pipelines

Both pipelines use `imblearn.pipeline.Pipeline` (not `sklearn.pipeline.Pipeline`) so that SMOTE is treated as a proper pipeline step: during cross-validation, imblearn only fits/applies SMOTE on the training fold of each split and leaves the validation fold at its natural imbalance.

- **Logistic Regression:** `StandardScaler` then `SMOTE` then `LogisticRegression` (LR is scale-sensitive, so `Amount`/`Time` need scaling before SMOTE interpolates neighbors).
- **Random Forest:** `SMOTE` then `RandomForestClassifier` (tree splits are scale-invariant, so no scaler).

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42

lr_pipeline = ImbPipeline(steps=[
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

rf_pipeline = ImbPipeline(steps=[
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("clf", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])

## 4. Hyperparameter tuning with GridSearchCV

Tuning is holistic: `smote__k_neighbors` is searched jointly with the classifier hyperparameters, inside the same cross-validated pipeline, so no combination of resampling and model settings is chosen using leaked validation information.

`StratifiedKFold` keeps each CV fold's class ratio representative of the training set. Scoring tracks Precision, Recall, and ROC-AUC together; refit selects the best model by ROC-AUC (a threshold-independent measure of separability), and precision/recall are still inspected per model below.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ["precision", "recall", "roc_auc"]

lr_param_grid = {
    "smote__k_neighbors": [3, 5, 7],
    "clf__C": [0.01, 0.1, 1, 10],
}

rf_param_grid = {
    "smote__k_neighbors": [3, 5, 7],
    "clf__n_estimators": [100, 200],
    "clf__max_depth": [5, 10, None],
}

lr_search = GridSearchCV(
    lr_pipeline, lr_param_grid,
    scoring=scoring, refit="roc_auc",
    cv=cv, n_jobs=-1, verbose=1
)

rf_search = GridSearchCV(
    rf_pipeline, rf_param_grid,
    scoring=scoring, refit="roc_auc",
    cv=cv, n_jobs=-1, verbose=1
)

In [ ]:
# Fit Logistic Regression grid search (SMOTE + scaling applied only within each training fold)
lr_search.fit(X_train, y_train)

print("Best LR params:", lr_search.best_params_)
print("Best LR CV ROC-AUC:", lr_search.best_score_)

In [ ]:
# Fit Random Forest grid search (SMOTE applied only within each training fold)
rf_search.fit(X_train, y_train)

print("Best RF params:", rf_search.best_params_)
print("Best RF CV ROC-AUC:", rf_search.best_score_)

## 5. Final evaluation on the untouched test set

`X_test`/`y_test` have never been seen by SMOTE, the scaler, or GridSearchCV, so they still reflect the real 99.83/0.17 imbalance. No accuracy metric: Precision, Recall, ROC-AUC, and a Confusion Matrix only.

In [ ]:
from sklearn.metrics import (
    precision_score, recall_score, roc_auc_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

def evaluate(name, fitted_search, X_test, y_test):
    best_model = fitted_search.best_estimator_
    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]

    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)

    print(f"=== {name} ===")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}")
    print()
    print(classification_report(y_test, y_pred, target_names=["Legitimate", "Fraud"]))

    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Legitimate", "Fraud"])
    disp.plot(cmap="Blues", values_format="d")
    plt.title(f"{name} - Confusion Matrix (test set)")
    plt.show()

    return {"model": name, "precision": precision, "recall": recall, "roc_auc": roc_auc}

lr_results = evaluate("Logistic Regression", lr_search, X_test, y_test)
rf_results = evaluate("Random Forest", rf_search, X_test, y_test)

In [ ]:
# Side-by-side comparison
results_df = pd.DataFrame([lr_results, rf_results]).set_index("model")
results_df